Все ячейки с выводом молекулы поменял в Markdown, так как иначе файлы занимают слишком много места для GitHub, результаты вывода орбиталей и Фукуи приложены отдельно в папке.

In [32]:
import numpy as np
from pyscf import gto, scf, cc, mp
from pyscf import dft
import py3Dmol
from pyscf.tools import cubegen

In [33]:
mol_xyz = '''4
Formaldehyde, B3LYP/6-31G(d,p) geometry 
  C           0.00000000       0.00000000       0.00000000
  O           1.20676000       0.00000000       0.00000000
  H          -0.59490414       0.93778067       0.00000000
  H          -0.59490414      -0.93778067       0.00000000
'''

view = py3Dmol.view(width=400, height=250)
view.addModel(mol_xyz, 'xyz')
view.setStyle({"stick":{"radius":0.15},"sphere": {"scale":0.25}})
#view.show()

In [34]:
mol =  gto.Mole()
mol.atom = '\n'.join(mol_xyz.split('\n')[2:])
mol.basis = 'def2-svp'
mol.build()

print('Число электронов (alpha, beta):', mol.nelec)
print('Число базисных функций:', mol.nao)

Число электронов (alpha, beta): (8, 8)
Число базисных функций: 38


In [35]:
mf = dft.RKS(mol, xc='B3LYP').run()
mf_hf    = scf.RHF(mol).run()
mpr      = mp.MP2(mf_hf).run()

converged SCF energy = -114.41531731019
converged SCF energy = -113.778111869256
E(RMP2) = -114.100027885406  E_corr = -0.321916016150258
E(SCS-RMP2) = -114.095247511477  E_corr = -0.317135642221527


In [36]:

KCALMOL = 627.5096080305927
print("Энергия образования в в методе B3LYP", (mf.e_tot*KCALMOL), " ккал/моль")
print("Энергия образования в в методе B3LYP", (mpr.e_tot*KCALMOL)," ккал/моль")
print("Энергия корреляции в методе B3LYP: ", (mf.e_tot - mf_hf.e_tot)*KCALMOL, " ккал/моль")
print("Энергия корреляции в методе MP2: ", mpr.e_corr*KCALMOL, " ккал/моль")

Энергия образования в в методе B3LYP -71796.71091801306  ккал/моль
Энергия образования в в методе B3LYP -71598.8637746508  ккал/моль
Энергия корреляции в методе B3LYP:  -399.8525364754711  ккал/моль
Энергия корреляции в методе MP2:  -202.00539311321816  ккал/моль


In [37]:
homo = mol.nelec[0] - 1
lumo = homo + 1

GRID = dict(nx=100, ny=100, nz=100, margin=6.0) 
cubegen.orbital(mol, 'h2co_homo.cube', mf.mo_coeff[:, homo], **GRID)
cubegen.orbital(mol, 'h2co_lumo.cube', mf.mo_coeff[:, lumo], **GRID)
homo_data = open('h2co_homo.cube', 'r').read()
lumo_data = open('h2co_lumo.cube', 'r').read()
print('done')

done


view = py3Dmol.view(width=600, height=250)
view.addModel(mol_xyz, 'xyz')
view.setStyle({"stick":{"radius":0.15},"sphere": {"scale":0.25}})
view.addVolumetricData(homo_data, 'cube',
                       {'isoval': -0.03, 'color': "red", 'opacity': 0.9})
view.addVolumetricData(homo_data, 'cube',
                       {'isoval': 0.03, 'color': "blue", 'opacity': 0.9})
#view.show()

view = py3Dmol.view(width=600, height=250)
view.addModel(mol_xyz, 'xyz')
view.setStyle({"stick":{"radius":0.15},"sphere": {"scale":0.25}})
view.addVolumetricData(lumo_data, 'cube',
                       {'isoval': -0.03, 'color': "cyan", 'opacity': 0.9})
view.addVolumetricData(lumo_data, 'cube',
                       {'isoval': 0.03, 'color': "orange", 'opacity': 0.9})
view.show()

In [38]:

mol_m = mol.copy()
mol_m.charge = 1
mol_m.spin = 1
mol_m.build()

mol_p = mol.copy()
mol_p.charge = -1
mol_p.spin = 1
mol_p.build()


In [39]:
mf_m = dft.UKS(mol_m, xc = 'B3LYP').run()
mf_p = dft.UKS(mol_p, xc = 'B3LYP').run()


WARN: HOMO -0.330104755512878 >= LUMO -0.330104755512878

converged SCF energy = -114.018839927766  <S^2> = 0.75834462  2S+1 = 2.0083273

WARN: HOMO -0.118401122965167 >= LUMO -0.118401122965166

converged SCF energy = -114.345915021361  <S^2> = 0.75298103  2S+1 = 2.0029788


In [40]:
dm = mf.make_rdm1() #матрица эл плотности
dm_m_alpha, dm_m_beta = mf_m.make_rdm1()# так как  UKS отдельные матрицы эл плотности, которые потом сами сложим
dm_m = dm_m_alpha + dm_m_beta
dm_p_alpha, dm_p_beta = mf_p.make_rdm1()
dm_p = dm_p_alpha + dm_p_beta

In [41]:
dm_cube = cubegen.density(mol, 'dm.cube', dm, **GRID)
dm_m_cube = cubegen.density(mol, 'dm_m.cube', dm_m, **GRID)
dm_p_cube = cubegen.density(mol, 'dm_p.cube', dm_p, **GRID)

In [42]:
temp_cube = cubegen.Cube(mol, **GRID)
temp_cube.write(dm_p_cube - dm_cube, "fukui_p.cube", comment='Fukui plus in real space (e/Bohr^3)')
temp_cube.write(dm_cube - dm_m_cube, "fukui_m.cube", comment='Fukui minus in real space (e/Bohr^3)')
temp_cube.write(0.5*(dm_p_cube - dm_m_cube), "fukui_0.cube", comment='Fukui 0 in real space (e/Bohr^3)')

In [43]:
fukui_p_data = open('fukui_p.cube', 'r').read()
fukui_m_data = open('fukui_m.cube', 'r').read()
fukui_0_data = open('fukui_0.cube', 'r').read()
print('done')

done


view = py3Dmol.view(width=600, height=250)
view.addModel(mol_xyz, 'xyz')
view.setStyle({"stick":{"radius":0.15},"sphere": {"scale":0.25}})
view.addVolumetricData(fukui_0_data, 'cube',
                       {'isoval': -0.005, 'color': "red", 'opacity': 0.9})
view.addVolumetricData(fukui_0_data, 'cube',
                       {'isoval': 0.005, 'color': "blue", 'opacity': 0.9})
#view.show()

view = py3Dmol.view(width=600, height=250)
view.addModel(mol_xyz, 'xyz')
view.setStyle({"stick":{"radius":0.15},"sphere": {"scale":0.25}})
view.addVolumetricData(fukui_p_data, 'cube',
                       {'isoval': -0.005, 'color': "red", 'opacity': 0.9})
view.addVolumetricData(fukui_p_data, 'cube',
                       {'isoval': 0.005, 'color': "blue", 'opacity': 0.9})
#view.show()

view = py3Dmol.view(width=600, height=250)
view.addModel(mol_xyz, 'xyz')
view.setStyle({"stick":{"radius":0.15},"sphere": {"scale":0.25}})
view.addVolumetricData(fukui_m_data, 'cube',
                       {'isoval': -0.005, 'color': "red", 'opacity': 0.9})
view.addVolumetricData(fukui_m_data, 'cube',
                       {'isoval': 0.005, 'color': "blue", 'opacity': 0.9})
#view.show()